# Week 4 Notebook: Teaching a Computer to Read

**AI for Beginners · Lesson 4 (Saturday + Sunday practice)**

In the slides you met tokens, word counts, TF-IDF, n-grams, similarity and embeddings, and saw how GPT-style models predict the next word. In this notebook you will **build small versions of these ideas yourself**.

### How to use this notebook
1. Run the cells **from top to bottom** (press `Shift + Enter` on each cell).
2. Wherever you see `# TODO`, replace the `____` blanks with your own code. A cell with blanks will show an error until you fill them in. That is normal.
3. Many tasks end with a **check** cell. If it prints a green tick, you got it right. If an `AssertionError` appears, read the message and try again.
4. Some questions ask you to **write an answer in words**. Double-click the markdown cell, type under **Your answer:**, and run the cell.
5. Stuck for more than 5 minutes? Re-read the hint, look at the cell above, then ask for help.

### What you will build
| Task | You will | Slide idea |
|---|---|---|
| 1 | Clean text without losing the word "not" | Tokens, cleaning |
| 2 | Count words and compute TF-IDF | Bag of words, TF-IDF |
| 3 | Train and judge a sentiment classifier | Supervised learning, metrics |
| 4 | See why word pairs (bigrams) help | N-grams |
| 5 | Build a bigram and trigram language model | Predicting the next word |
| 6 | Measure text similarity and build a tiny search engine | Cosine similarity |
| 7 | Bonus: see what a speech model "sees" | ASR and TTS |

In [ ]:
# Run this cell first. It loads the tools we need.
import re
import math
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

def ok(message="Correct!"):
    print("✅", message)

print("Tools loaded.")

## The data: 60 short restaurant reviews

Here are 60 made-up reviews, half positive and half negative. Look at them carefully. Notice the little phrases **"not bad"** (which is positive!) and **"not good"** (which is negative). These small phrases will cause trouble later, and that is the point.

*The reviews are invented for teaching. A real project would use thousands of real reviews.*

In [ ]:
positive = [
    "The food was delicious and the staff were friendly",
    "Great service and a lovely atmosphere",
    "I loved the pasta, it was fresh and tasty",
    "Not bad at all, I would come back",
    "The pizza was amazing, best in town",
    "Fast delivery and the food arrived hot and fresh",
    "Really enjoyed our dinner, great value",
    "Friendly waiter, generous portions and tasty food",
    "Not bad, the soup was warm and tasty",
    "A wonderful meal, I highly recommend it",
    "Clean tables, quick service and lovely desserts",
    "The coffee was excellent and the cake was great",
    "Not bad for the price, the rice was good",
    "Friendly team and fresh ingredients",
    "I was happy with my order, it arrived on time",
    "Our family had a great evening, thank you",
    "Not bad, the burger was juicy and big",
    "Delicious breakfast, I will be back soon",
    "The fish was perfect and the sauce was wonderful",
    "Superb experience, the staff were very helpful",
    "Good food, good service and a lovely place",
    "The pasta was good and the waiter was nice",
    "Very good breakfast, the coffee was great",
    "Great pizza, friendly staff and fast delivery",
    "I really enjoyed the soup, it was fresh and delicious",
    "Wonderful staff, they were kind and helpful",
    "The dessert was amazing and the portions were generous",
    "Perfect dinner, delicious fish and lovely service",
    "Good value, tasty food and a happy atmosphere",
    "Fresh salad, great sauce and quick friendly service",
]
negative = [
    "The food was cold and the staff were rude",
    "Terrible service and a dirty table",
    "I hated the pasta, it was bland and soggy",
    "Not good at all, I would not come back",
    "The pizza was awful, worst in town",
    "Late delivery and the food arrived cold",
    "Really disappointed with our dinner, poor value",
    "Rude waiter, tiny portions and bland food",
    "Not good, the soup was cold and salty",
    "A horrible meal, I do not recommend it",
    "Dirty tables, slow service and stale desserts",
    "The coffee was bitter and the cake was dry",
    "Not good for the price, the rice was hard",
    "Unfriendly team and old ingredients",
    "I was unhappy with my order, it arrived late",
    "Our family had a bad evening, never again",
    "Not good, the burger was greasy and small",
    "Awful breakfast, I will not be back",
    "The fish was overcooked and the sauce was bland",
    "Poor experience, the staff were not helpful",
    "Bad food, bad service and a dirty place",
    "The pasta was bad and the waiter was rude",
    "Very bad breakfast, the coffee was cold",
    "Bad pizza, rude staff and late delivery",
    "I really disliked the soup, it was cold and salty",
    "Terrible staff, they were slow and rude",
    "The dessert was awful and the portions were tiny",
    "Worst dinner, greasy fish and dirty service",
    "Poor value, bland food and a miserable atmosphere",
    "Soggy salad, bad sauce and slow rude service",
]

df = pd.DataFrame({
    "text": positive + negative,
    "label": ["positive"] * len(positive) + ["negative"] * len(negative),
})
print(df["label"].value_counts())
df.sample(5, random_state=1)

---
## Task 1: Clean the text

Computers cannot read sentences. First we turn each review into a list of **tokens** (words). Cleaning usually has four small steps:

1. **Lowercase** everything, so "Great" and "great" count as the same word.
2. **Remove punctuation** such as commas and exclamation marks.
3. **Split** the text into words (tokens).
4. **Remove stop words**: very common words such as "the" and "was" that carry little meaning.

**Careful.** Many stop-word lists also contain "not", "no" and "never". Remove those and **"not good" becomes "good"**, which reverses the meaning! So in your cleaning function, you must **keep the negation words**.

**Hints**
- `text.lower()` makes lowercase.
- `re.sub(r"[^a-z0-9 ]", "", text)` keeps only letters, digits and spaces.
- `text.split()` splits on spaces.
- A list comprehension can filter: `[t for t in tokens if t not in SOME_SET]`.

In [ ]:
STOP_WORDS = {"the", "a", "an", "is", "was", "were", "and", "of", "to", "it", "this", "that",
              "i", "we", "in", "on", "at", "for", "with", "be", "are", "as", "our", "my",
              "not", "no", "never", "nor"}          # note: "not", "no", "never" are in here!
NEGATIONS = {"not", "no", "never"}

def clean_text(text):
    # TODO 1a: make the text lowercase
    text = ____
    # TODO 1b: remove punctuation (keep only letters, digits and spaces)
    text = ____
    # TODO 1c: split the text into a list of tokens
    tokens = ____
    # TODO 1d: remove stop words, BUT keep the words in NEGATIONS
    tokens = [t for t in tokens if ____]
    return tokens

print(clean_text("The food was NOT good!"))

In [ ]:
# Check for Task 1
assert clean_text("The food was NOT good!") == ["food", "not", "good"], "Expected ['food', 'not', 'good']. Did you keep 'not'?"
assert clean_text("Great, great SERVICE.") == ["great", "great", "service"], "Check lowercase and punctuation removal"
ok("Task 1 passed. Your cleaner keeps 'not'.")

**Question 1.** In your own words: why would removing the word "not" be a problem for a sentiment tool? Give an example sentence.

**Your answer:**

In [ ]:
# Apply your cleaner to every review. (Nothing to change here.)
df["tokens"] = df["text"].apply(clean_text)
df["clean"] = df["tokens"].apply(" ".join)
df[["text", "clean"]].head(5)

---
## Task 2: Count words and weigh them (bag of words and TF-IDF)

A **bag of words** counts how often each word appears and ignores the order. **TF-IDF** improves on counting: a word gets a **high** weight if it is **frequent in this review** but **rare across all reviews**. Words such as "the" appear everywhere, so they get a low weight.

**2a.** Count the words in all **negative** reviews and print the 5 most common. `Counter` can do the counting; `.most_common(5)` gives the top 5.

In [ ]:
negative_tokens = []
for tokens in df[df["label"] == "negative"]["tokens"]:
    negative_tokens.extend(tokens)

# TODO 2a: build a Counter from negative_tokens and print its 5 most common words
counts = ____
print(____)

**2b.** Turn all the cleaned reviews into a **TF-IDF matrix**. Each row is a review, each column is a word, and each number is that word's weight in that review.

`TfidfVectorizer()` has two main methods: `.fit_transform(list_of_texts)` learns the vocabulary and returns the matrix.

In [ ]:
# TODO 2b: create a TfidfVectorizer and use fit_transform on df["clean"]
vec = ____
X = ____
print("Matrix shape (reviews x words):", X.shape)

**2c.** Which words matter most in **positive** reviews? Take the average TF-IDF weight of each word over the positive reviews and print the top 5 words.

*Hints:* `np.where(condition)[0]` gives row numbers; `X[rows].mean(axis=0)` averages those rows; `.argsort()[::-1]` sorts from largest to smallest; `vec.get_feature_names_out()` gives the words.

In [ ]:
# TODO 2c
positive_rows = np.where(df["label"] == ____)[0]
mean_weights = np.asarray(X[____].mean(axis=0)).ravel()
top_idx = mean_weights.argsort()[::-1][:5]
words = vec.get_feature_names_out()
print([words[i] for i in top_idx])

In [ ]:
# Check for Task 2
assert X.shape[0] == 60, "There should be one row per review (60)"
assert counts.most_common(1)[0][1] >= 5, "Counter looks wrong: did you count the negative tokens?"
assert len(top_idx) == 5
ok("Task 2 passed.")

**Question 2.** The words in your list for 2c: are they what you expected? Is there a word that surprised you? Why might a word be frequent in positive reviews but get only a medium TF-IDF weight?

**Your answer:**

---
## Task 3: Train a sentiment classifier

Now the **supervised learning** step. We have the answers (labels), so we can train a model that reads a review and says *positive* or *negative*.

The recipe from Week 3:
1. **Split** the data into training and test sets (the test set is kept hidden until the end).
2. **Train** on the training set.
3. **Predict** on the test set.
4. **Judge** with accuracy, precision and recall.

We use a **pipeline**: TF-IDF first, then logistic regression (a classification model despite its name). We set `C=10`, which makes the model a little less cautious. That helps when the dataset is as small as ours.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["label"], test_size=0.25, random_state=1, stratify=df["label"]
)
print("Training reviews:", len(X_train), " Test reviews:", len(X_test))

# TODO 3a: build a pipeline with TfidfVectorizer() first and LogisticRegression(C=10) second
model = make_pipeline(____, ____)

# TODO 3b: train the model on the training data
model.fit(____, ____)

# TODO 3c: predict the labels of the test reviews
pred = model.predict(____)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

**3d.** Look at the **confusion matrix**. Rows are the true labels, columns are the predicted labels (in alphabetical order: negative, positive).

In [ ]:
labels = ["negative", "positive"]
cm = confusion_matrix(y_test, pred, labels=labels)
print(pd.DataFrame(cm, index=["true negative", "true positive"], columns=["predicted negative", "predicted positive"]))

In [ ]:
# Check for Task 3
assert len(pred) == len(y_test), "You need one prediction per test review"
assert accuracy_score(y_test, pred) > 0.5, "Your model is no better than guessing. Check the pipeline"
ok("Task 3 passed.")

**Question 3.** Look at your results. (a) Is accuracy a good number to rely on here? (b) With only 15 test reviews, how much should you trust the result? (c) If this model flagged *negative* reviews for a manager to read, which would be worse: missing a negative review, or flagging a positive one? Which metric would you watch, precision or recall?

**Your answer:**

Now try the model on **your own sentences**. Change the examples and run again. Remember that the model sees the cleaned text, so use `clean_text` first.

In [ ]:
my_sentences = [
    "The staff were wonderful and the food was great",
    "Terrible, cold food and slow service",
]
for s in my_sentences:
    cleaned = " ".join(clean_text(s))
    prob_pos = model.predict_proba([cleaned])[0][list(model.classes_).index("positive")]
    print(f"{s!r}\n   -> {model.predict([cleaned])[0]} (probability positive: {prob_pos:.2f})")

---
## Task 4: Why word pairs help (unigrams and bigrams)

So far each word is looked at **alone** (a *unigram*). But meaning often lives in **pairs**: "not good" is negative, "not bad" is positive. A **bigram** is a pair of neighbouring words.

Let us see what happens with:
- **A. Bad cleaning:** we remove *every* stop word, including "not".
- **B. Careful cleaning, unigrams only:** the pipeline you just built.
- **C. Careful cleaning, unigrams + bigrams:** `TfidfVectorizer(ngram_range=(1, 2))` looks at single words **and** pairs.

In [ ]:
def clean_text_bad(text):
    """Removes ALL stop words, including 'not'."""
    text = re.sub(r"[^a-z0-9 ]", "", text.lower())
    return [t for t in text.split() if t not in STOP_WORDS]

df["clean_bad"] = df["text"].apply(clean_text_bad).apply(" ".join)
print(df.loc[3, "text"], "->", df.loc[3, "clean_bad"])   # review 3 is "Not bad at all, I would come back"

**4a.** Build model **C**: the same pipeline as before, but give `TfidfVectorizer` the setting `ngram_range=(1, 2)`. Then compare all three on tricky sentences.

For a fair comparison we train all three models on **all 60 reviews** this time (it is a demo, not a test).

In [ ]:
# Model A: bad cleaning, unigrams
model_a = make_pipeline(TfidfVectorizer(), LogisticRegression(C=10))
model_a.fit(df["clean_bad"], df["label"])

# Model B: careful cleaning, unigrams
model_b = make_pipeline(TfidfVectorizer(), LogisticRegression(C=10))
model_b.fit(df["clean"], df["label"])

# TODO 4a: Model C: careful cleaning, unigrams AND bigrams  (use ngram_range=(1, 2))
model_c = make_pipeline(TfidfVectorizer(____), LogisticRegression(C=10))
model_c.fit(df["clean"], df["label"])

In [ ]:
tricky = [
    "The food was good",
    "The food was bad",
    "The food was not good",
    "The food was not bad",
    "Not bad at all",
    "The service was not helpful",
]

def prob_positive(m, text, bad=False):
    cleaned = " ".join(clean_text_bad(text) if bad else clean_text(text))
    return m.predict_proba([cleaned])[0][list(m.classes_).index("positive")]

rows = []
for s in tricky:
    rows.append({
        "sentence": s,
        "A: bad cleaning": round(prob_positive(model_a, s, bad=True), 2),
        "B: unigrams": round(prob_positive(model_b, s), 2),
        "C: unigrams + bigrams": round(prob_positive(model_c, s), 2),
    })
pd.DataFrame(rows)

In [ ]:
# Check for Task 4
assert model_c.named_steps["tfidfvectorizer"].ngram_range == (1, 2), "Model C should use ngram_range=(1, 2)"
ok("Task 4 passed. Now read the table above carefully.")

**Question 4.** The table shows the model's **probability of "positive"** for each sentence (close to 1 means very positive, close to 0 means very negative). For "The food was not good" the right answer is negative and for "The food was not bad" it is positive.

(a) Which of the three models gives the most sensible probabilities for the tricky sentences? Which is the weakest?
(b) Why does throwing away "not" hurt model A?
(c) Why can bigrams help model C?
(d) The training set had only 60 reviews. Would you trust any of these models in real life? What would you change?

**Your answer:**

---
## Task 5: Build a tiny language model (bigrams and trigrams)

This is the **same idea your phone keyboard uses** to suggest the next word, and a (very small) ancestor of GPT:

> Count how often each word follows each other word. Turn the counts into probabilities. Pick the most likely next word.

For a bigram model:  P(next word | previous word) = count(previous, next) ÷ count(previous)

We will start with the tiny text from the quiz:

```
I like pizza
I like pasta
I eat pizza
```

**5a.** Write a function `count_ngrams(sentences, n)` that counts every group of `n` consecutive words. Each sentence is a list of words, and you **should not count across sentences**.

*Hint:* for a sentence with words `w`, the groups are `tuple(w[i:i+n])` for `i` from 0 to `len(w) - n`.

In [ ]:
tiny = [s.split() for s in ["i like pizza", "i like pasta", "i eat pizza"]]

def count_ngrams(sentences, n):
    counts = Counter()
    for words in sentences:
        # TODO 5a: for every starting position i, add the n-word group to counts
        for i in range(____):
            gram = ____
            counts[gram] += 1
    return counts

bigrams = count_ngrams(tiny, 2)
print(bigrams)

**5b.** Now the probability. `prob_next(counts_n, counts_n_minus_1, context, word)` returns the probability of `word` after `context`.

- `context` is a tuple of the previous words, for example `("like",)` for a bigram model or `("i", "like")` for a trigram model.
- Probability = count of (context + word) ÷ count of context. If the context was never seen, return 0.

In [ ]:
unigrams = count_ngrams(tiny, 1)

def prob_next(counts_n, counts_context, context, word):
    # TODO 5b: look up count(context + word) and count(context), then divide
    numerator = counts_n[____]
    denominator = counts_context[____]
    if denominator == 0:
        return 0.0
    return numerator / denominator

print("P(pasta | like) =", prob_next(bigrams, unigrams, ("like",), "pasta"))

In [ ]:
# Check for Task 5a and 5b
assert bigrams[("i", "like")] == 2, "'i like' appears twice"
assert abs(prob_next(bigrams, unigrams, ("like",), "pasta") - 0.5) < 1e-9, "P(pasta | like) should be 1/2"
assert abs(prob_next(bigrams, unigrams, ("i",), "like") - 2/3) < 1e-9, "P(like | i) should be 2/3"
assert prob_next(bigrams, unigrams, ("like",), "sushi") == 0.0, "An unseen pair has probability 0"
ok("Tasks 5a and 5b passed.")

**5c.** Suggest the next word. Write `suggest(counts_n, counts_context, context, k=3)` which returns the `k` most likely next words with their probabilities.

*Hint:* loop over the n-grams in `counts_n` whose first words equal `context`, compute their probability, and sort from largest to smallest.

In [ ]:
def suggest(counts_n, counts_context, context, k=3):
    context = tuple(context)
    results = []
    for gram in counts_n:
        # TODO 5c: if gram starts with the context, record (last word, probability)
        if gram[:-1] == ____:
            word = ____
            results.append((word, prob_next(counts_n, counts_context, context, word)))
    results.sort(key=lambda pair: pair[1], reverse=True)
    return results[:k]

print(suggest(bigrams, unigrams, ("i",)))

In [ ]:
# Check for Task 5c
top = suggest(bigrams, unigrams, ("i",))
assert top[0][0] == "like" and abs(top[0][1] - 2/3) < 1e-9, "After 'i' the best word is 'like' with probability 2/3"
assert suggest(bigrams, unigrams, ("eat",))[0][0] == "pizza"
ok("Task 5c passed.")

### Now try it on the real reviews

Let's train the same model on all 60 reviews (lowercase, punctuation removed, but **no stop-word removal**, because a language model needs the full sentences). Then we compare **bigrams** (one word of memory) with **trigrams** (two words of memory).

In [ ]:
sentences = [re.sub(r"[^a-z0-9 ]", "", t.lower()).split() for t in df["text"]]

uni = count_ngrams(sentences, 1)
bi = count_ngrams(sentences, 2)
tri = count_ngrams(sentences, 3)
print("Different words:", len(uni), "| different bigrams:", len(bi), "| different trigrams:", len(tri))

print("\nAfter 'not' (bigram):      ", suggest(bi, uni, ("not",), k=4))
print("After 'was' (bigram):      ", suggest(bi, uni, ("was",), k=4))

**5d.** Do the same with a **trigram** model: the context is now **two** words, for example `("were", "not")`. Print the top suggestions after `("were", "not")` and after `("i", "will")`.

In [ ]:
# TODO 5d: use suggest() with the trigram counts (tri) and the bigram counts (bi) as the context counts
print("After 'were not' (trigram):", suggest(____, ____, ("were", "not"), k=4))
print("After 'i will' (trigram): ", suggest(____, ____, ("i", "will"), k=4))

In [ ]:
# Check for Task 5d
assert len(suggest(tri, bi, ("i", "will"), k=4)) > 0, "Trigram suggestions should not be empty. Use tri, then bi"
ok("Task 5d passed.")

**Question 5.**
(a) Compare the bigram and trigram suggestions. Which model is more certain? Which one has seen less data for each context?
(b) What happens if you ask for suggestions after a word that is not in the reviews, such as `("sushi",)`? Try it in a new cell. What does this tell you about n-gram models?
(c) What would happen to the number of different n-grams if we went from trigrams to 5-grams on a book with a million words? Why is that a problem?
(d) In one sentence: what can GPT-style models do that your bigram model cannot?

**Your answer:**

---
## Task 6: Similarity and a tiny search engine

To compare two texts we turn each into a vector of numbers, then measure the angle between them. **Cosine similarity** equals 1 when two vectors point the same way and 0 when they have nothing in common.

cosine(A, B) = (A · B) ÷ (|A| × |B|)

**6a.** Complete `cosine` using NumPy: `np.dot(a, b)` is the dot product and `np.linalg.norm(a)` is the length of a vector.

In [ ]:
def cosine(a, b):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    # TODO 6a: return the dot product divided by the product of the two lengths
    return ____

# Vocabulary: [I, like, pizza, pasta]
doc_a = [1, 1, 1, 0]   # "I like pizza"
doc_b = [1, 1, 0, 1]   # "I like pasta"
print("cosine(A, B) =", round(cosine(doc_a, doc_b), 3))

In [ ]:
# Check for Task 6a
assert abs(cosine(doc_a, doc_b) - 2/3) < 1e-9, "Expected 2/3 (about 0.667)"
assert abs(cosine([2, 0, 3], [4, 0, 6]) - 1.0) < 1e-9, "Same direction means similarity 1"
ok("Task 6a passed. (Same direction gives 1, even when one vector is twice as long.)")

**6b. A tiny help-centre search engine.** Below are six help articles. We turn them into TF-IDF vectors. When a user types a question, we turn it into a vector the same way and rank the articles by cosine similarity.

*Hint:* `vec.transform([query])` converts a query using the vocabulary the vectorizer already learned. `cosine_similarity(a, b)` from scikit-learn compares two matrices.

In [ ]:
articles = [
    "How to reset your password",
    "Troubleshooting failed payments and card errors",
    "How to change your delivery address",
    "Refund policy for damaged items",
    "Opening hours and contact details",
    "How to track your order after it ships",
]

search_vec = TfidfVectorizer(stop_words="english")
article_matrix = search_vec.fit_transform(articles)

def search(query, top=3):
    # TODO 6b: turn the query into a vector, compare it with all articles, and sort the best first
    q_vec = search_vec.____([query])
    scores = cosine_similarity(q_vec, article_matrix).ravel()
    order = scores.argsort()[::-1][:top]
    return [(articles[i], round(float(scores[i]), 2)) for i in order]

for line in search("my card payment failed"):
    print(line)

In [ ]:
# Check for Task 6b
best = search("my card payment failed", top=1)[0][0]
assert best == "Troubleshooting failed payments and card errors", "The payments article should rank first"
assert search("reset password")[0][0] == "How to reset your password"
ok("Task 6b passed.")

**6c.** Now try a question that shares **no words** with the right article. Run the cell and look at the scores.

In [ ]:
print(search("the shop took my money twice"))
print(search("where is my parcel"))

**Question 6.**
(a) For "the shop took my money twice" the right article is probably "Refund policy for damaged items" or the payments article, but the scores are all 0 or very low. Why does TF-IDF fail here?
(b) What kind of model would fix this? (Think back to the slides: words as points on a map.)
(c) Why do we use **cosine similarity** and not just the distance between two count vectors?

**Your answer:**

---
## Summary

You built, from scratch or nearly so:

- a **text cleaner** that protects the word "not"
- **bag-of-words** counts and **TF-IDF** weights
- a **sentiment classifier**, judged on unseen data
- an experiment showing why **bigrams** help
- a **bigram and trigram language model** (the tiny ancestor of GPT)
- **cosine similarity** and a small **search engine**
- (bonus) a **spectrogram**, the picture speech models read

### Self-check: can you explain these in one sentence each?
- Why must we test on data the model has not seen?
- Why is removing "not" dangerous?
- What is the difference between a bigram and a trigram, and the cost of using longer n-grams?
- Why does cosine similarity ignore the length of a document?
- Why do embeddings find matches that TF-IDF misses?
- What does ASR turn into what, and what does TTS turn into what?

**Next:** Week 5 covers current trends and AI agents. In Week 6 there is a revision session and the final exam.